# Playground Series S6E9 — 「S6E9 Single XGB CV: 0.94488」解説付き写し

| | |
|---|---|
| **コンペ** | [Predicting Electric Vehicle Purchases (Playground S6E9)](https://www.kaggle.com/competitions/playground-series-s6e9) |
| **元notebook** | [S6E9 Single XGB CV: 0.94488](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94488) |
| **原著者** | Дворкин Евгений Владимирович（`evgendvorkin`） |
| **スコア** | OOF AUC **0.94488** / Public LB **0.94460**（49 votes = S6E9で最多得票） |
| **日付** | 2026-09-05 |

> ⚠️ **これは学習目的の解説付き写しです。** コードは原著のものを一字一句そのまま保持し、出力と実行番号だけを消しています。実行はしていません。原著者への敬意として、必ず元notebookにupvoteしてください。
>
> なお冒頭のセルは `from Dvorkin_style_engine import *` という原著者オリジナルの表示用ライブラリに依存しています（`h1`, `h2`, `info_card`, `pretty_df`, `md_metric` はすべて見た目を整えるだけの関数です）。手元で動かす場合は、それらを `print` に置き換えれば本質は変わりません。

---

## なぜこのnotebookを選んだか

S6E9のLB最上位（0.94633）は、実は**OOF予測のCSVを共有するだけのnotebook**で、手法としては学べるものがほとんどありません。その下の 0.9462 帯もほぼ全てが「公開notebookのsubmissionをランク平均する」ブレンドです。

対してこのnotebookは、**単一のXGBoostモデル**で 0.9446 まで到達しており、しかも**各施策の効果を1つずつ測って表にしている**——つまり `ablation study`（要素を1つずつ足し引きして寄与を測る実験）が最初から書かれています。「効いた」と主張する前に測る、という作法を学ぶ教材として、公開notebookの中で最も価値が高いと判断しました。

| 施策 | OOF AUC | 差分 |
|---|---|---|
| ベースライン | 0.94204 | — |
| + digit features | 0.94347 | **+0.00143** |
| + frequency encoding | 0.94466 | **+0.00119** |
| + ハイパーパラメータ更新 | 0.94488 | +0.00022 |
| + 特徴量選択（154→76） | 0.94488 | ±0.00000（速くなっただけ） |

さらに「10-fold vs 5-fold」「固定シード vs 動的シード」も測って、**どちらも差がなかった**と正直に書いています。**効かなかった実験を書き残す**のは、効いた実験を書くより価値があります。

---

## 評価指標

**タスク**：顧客属性（年齢・年収・通勤距離・充電スタンドの数・環境意識レベルなど）から、**その人が電気自動車を買うか（`Will_Buy_EV`）** を予測する二値分類。合成データ（synthetic data）で作られた Playground コンペです。

**指標**：**ROC-AUC**。全ての「正例1つ・負例1つ」のペアについて、正例のスコアの方が高い割合。

この指標の決定的な性質は**順位不変（rank-invariant）**であることです：

- 予測値をどんな単調増加関数で変換しても、AUCは**1ビットも変わりません**。つまり**確率のキャリブレーション（0.7という予測が本当に70%になるよう補正すること）だけでは、原理的に1点も上がりません**。原著が「Isotonic, Rank, Clip の後処理を試したが改善なし」と書いているのは、この性質からすれば当然の結果です。
- 逆に、AUCを上げられるのは**順位を実際に入れ替える操作**だけ——新しい特徴量、別のアーキテクチャ、モデル間ブレンド。
- また**しきい値に依存せず、クラス不均衡にも比較的安定**なので、正例率が偏りがちな購買予測に向いています。

**このnotebookが指標をどう最適化しているか**：**新しい特徴量で順位を作り変える**ことに集中しています。digit features（56個）と frequency encoding（全列）はどちらも「元の値の単調変換」では**ない**——桁の抽出も出現頻度への置換も、値の大小関係を壊す非単調な変換です。だからこそAUCが動きます。後処理（キャリブレーション）は単調変換なので効かない。**指標の性質から、どこに労力を割くべきかが導かれている**好例です。


## 📈 Experiments & Results

### 🔢 Digit Features — Win

OOF AUC improved from **0.94204** to **0.94347** (+0.00143).

| Metric | Before digit | After digit | Gain |
|--------|--------------|-------------|------|
| OOF AUC | 0.94204 | **0.94347** | **+0.00143** |

Extracting 56 digit features from 7 numerical columns helped the model capture synthetic generation artifacts. XGBoost now uses many new uncorrelated signals, improving overall ranking.

---

### 📊 Frequency Encoding — Win

OOF AUC improved from **0.94347** to **0.94466** (+0.00119).

| Metric | Before Frequency | After Frequency | Gain |
|--------|------------------|-----------------|------|
| OOF AUC | 0.94347 | **0.94466** | **+0.00119** |

Frequency encoding adds the normalized occurrence rate of each value across train+test. This helps the model identify rare and common patterns, and exposes subtle distributional signals in numerical and digit features that raw values do not reveal.

---

### 🎯 Magic Features — No Effect

4 synthetic-artifact flags were tested and removed. No improvement was observed.

| Metric | Before Magic | After Magic | Change |
|--------|--------------|-------------|--------|
| OOF AUC | **0.94466** | 0.94465 | -0.00001 |

The four flags (`is_30k_spike`, `is_millionaire_cliff`, `is_dead_zone`, `is_env_hater`) captured deterministic synthetic-data artifacts, but XGBoost did not benefit from them.

---

### 🎯 Original Dataset Target Means — Slight Decrease

Added original-dataset target statistics for all features. XGBoost did not improve.

---

### 📂 Numeric-to-Categorical + Frequency — No Effect

Converted all numeric features (including digit features) into string categories and added frequency encoding for them.

| Metric | Before | After | Change |
|--------|--------|-------|--------|
| OOF AUC | **0.94466** | 0.94459 | -0.00007 |

This approach was inspired by the top LightGBM solution (0.94587), but XGBoost did not benefit from this representation.

---

### 🚀 Hyperparameter Upgrade — Win

Replaced the old fast-training parameters with a GPU-accelerated, deeply-regularized setup inspired by top LightGBM solutions.

| Parameter | Before | After | Why |
|-----------|--------|-------|-----|
| learning_rate | 0.03 | **0.005** | Slower but deeper, more stable learning |
| n_estimators | 2000 | **10000** | More room to learn with low LR |
| min_child_weight | 5 | **10** | Stronger regularization against noise |
| subsample | 0.7 | **0.9** | More rows per tree, less variance |
| colsample_bytree | 0.5 | **0.9** | More features per split, richer signal |
| device | cpu | **cuda** | 3-5x faster training on GPU |
| reg_alpha | — | **0.071** | L1 regularization for feature selection |
| reg_lambda | — | **2.0** | L2 regularization for stable leaf weights |
| max_bin | 256 | **1024** | Better split resolution |
| early_stopping_rounds | 100 | **700** | Higher patience for slow learning |

**Result:** OOF AUC improved from **0.94466** to **0.94488** (+0.00022).

---

### 🧹 Feature Selection — Cleaner Model, Same Score

Removed constant and perfectly-correlated features. The model became faster and simpler while keeping the same performance.

| Metric | Before (154 features) | After (76 features) | Change |
|--------|----------------------|---------------------|--------|
| OOF AUC | 0.94488 | **0.94488** | 0.00000 |
| LB | 0.94460 | **0.94460** | 0.00000 |
| Features | 154 | **76** | -78 |

Post-processing tests (Isotonic, Rank, Clip) were also evaluated — no improvement. The cleaner 76-feature model is the final version.

## 🔬 Experiment: 5-fold vs 10-fold CV

Tested whether switching from 10 folds to 5 folds (like top solutions) improves the leaderboard score.

| Metric | 10-fold | 5-fold | Change |
|--------|---------|--------|--------|
| OOF AUC | **0.94488** | 0.94470 | -0.00018 |
| LB | **0.94460** | 0.94459 | -0.00001 |

**Conclusion:** Fold count does not affect leaderboard performance. 10-fold CV remains our validation strategy.

## 🔬 Experiment: Fixed Seed vs Dynamic Seeds

Tested whether using a single fixed seed (42) instead of dynamic per-fold seeds (42+fold) affects the leaderboard score.

| Metric | Dynamic Seeds (42+fold) | Fixed Seed (42) | Change |
|--------|-------------------------|-----------------|--------|
| OOF AUC | **0.94488** | 0.94487 | -0.00001 |
| LB | **0.94460** | 0.94461 | +0.00001 |

**Conclusion:** Seed strategy does not affect performance. Dynamic seeds remain our validation approach. Final model selection will be based on the best CV score, not the public leaderboard.

All experiments are conducted based on notebooks:

https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94587-lb-0-94612

https://www.kaggle.com/code/lucifer19/ev-quantum-forge-s6e9-xgboost?scriptVersionId=346769553

https://www.kaggle.com/code/kirill0212/s6e9-lightgbm

https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights

https://www.kaggle.com/code/mikhailnaumov/electric-vehicle-purchases-xgb

## 📦 1. IMPORTS + SETUP

### セル解説：インポートと計測用ロガー

**何をしているか**：ライブラリを読み込み、`log()` という「起動からの経過秒数つきprint」を定義しています。

**なぜそうするのか**：`T0 = time.perf_counter()` を基準に経過時間を出しておくと、後で「どのfoldに何秒かかったか」がログから復元できます。学習が想定より遅いとき、原因の特定がずっと楽になります。`flush=True` は、Kaggleのログにその場で書き出させるための指定です（付けないとバッファに溜まって、途中で落ちたときに何も残りません）。

`sys.path.append(...)` して `Dvorkin_style_engine` を読み込んでいるのは、**原著者が自作した表示スタイルのライブラリ**です。`h1`/`h2`/`info_card`/`pretty_df` は装飾用の関数で、モデルの精度には一切関係しません。

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/evgendvorkin/dvorkin-visual-v1/')
from Dvorkin_style_engine import *

import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy import stats

warnings.filterwarnings('ignore')

T0 = time.perf_counter()
def log(msg):
    print(f'[{time.perf_counter() - T0:7.1f}s] {msg}', flush=True)

h1("🚗 EV Adoption Prediction — S6E9")
info_card("✅ Setup Complete", "Импорты выполнены, стиль загружен.", style="vi")

## 📥 2. LOAD DATA

### セル解説：データの読み込み

**何をしているか**：`train.csv`、`test.csv` に加えて、**`orig`（オリジナルデータセット）**を読み込んでいます。

**なぜそうするのか**：Playground Series のデータは、実在するデータセットを元に**深層生成モデルで合成された**ものです。Kaggleは元データセットへのリンクも公開しており、多くの上位解法は元データを学習に追加します（データが増える＋合成過程で失われた本物の相関が補える）。

ただし注意：このnotebookでは `orig` を読み込んではいますが、**この後の処理で実際には使っていません**。読み込んだけれど活かしていない、という状態です。これは改善余地の一つで、READMEの改善提案にも記載しています。**「読み込んだ＝使った」ではない**ことを、コードを読むときは意識してください。

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv')
orig = pd.read_csv('/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv')

h2("📥 Data Loading")
info_card("Dataset Shapes", 
          f"Train: {train.shape} | Test: {test.shape} | Orig: {orig.shape}", 
          style="vi")
pretty_df(train.head(3))

## 🔧 3. PREPROCESSING (BASE)

### セル解説：前処理（ラベルエンコーディング）

**何をしているか**：カテゴリ列（性別・都市タイプ・現在の車種・自宅充電可否・補助金の有無・航続距離不安レベル）を、`{値: 連番}` の辞書で整数に置き換えています。

**なぜそうするのか**：XGBoostのような**決定木ベースのモデル**は、カテゴリを整数に置き換えるだけで扱えます。線形モデルなら「1と2の差は2と3の差と同じ」という誤った順序が問題になるのでone-hotが必要ですが、木は「値が2.5より大きいか」という分割を繰り返すだけなので、**任意の分割グループを表現できます**（深さが十分あれば）。one-hotより列数が爆発しないので、木モデルではこちらが定石です。

**注意点**：`mapping` を **train だけ**から作っています（`train[col].unique()`）。test にしか出てこない値があれば `NaN` になります。今回は合成データで値の集合が一致しているので問題ありませんが、実務では `pd.concat([train, test])` から作るか、未知値用の枠を用意するのが安全です。

**用語補足**：*ラベルエンコーディング* = カテゴリを整数に置換すること。*one-hot エンコーディング* = カテゴリ数だけ 0/1 の列を作ること。

In [ ]:
cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
            'Subsidy_Available', 'Range_Anxiety_Level']

for col in cat_cols:
    mapping = {val: i for i, val in enumerate(train[col].unique())}
    train[f'LE_{col}'] = train[col].map(mapping)
    test[f'LE_{col}'] = test[col].map(mapping)

# Удаляем исходные категориальные колонки
train = train.drop(columns=cat_cols)
test = test.drop(columns=cat_cols)

# Убираем id
train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

# Разделяем X и y
X = train.drop(columns=['Will_Buy_EV'])
y = train['Will_Buy_EV']

h2("🔧 Preprocessing")
info_card("✅ Base Preprocessing Done", f"X shape: {X.shape} | Test shape: {test.shape}", style="vi")

## 🧪 4. FEATURE ENGINEERING — Powerful Interactions

### セル解説：交互作用特徴量（interaction features）

**何をしているか**：EDA（探索的データ分析）で見つけた組み合わせを、**掛け算した新しい列**として追加しています。例：`環境意識レベル × 補助金の有無`、`環境意識 × 年収`、`補助金 × 年収`、`環境意識 × 通勤距離`。

**なぜそうするのか**：決定木は原理的に交互作用を表現できます（「環境意識が高い」で分割 → その中でさらに「補助金あり」で分割）。しかしそれには**2段階の分割**が必要で、各分割で使えるデータが半分になっていきます。**掛け算した列をあらかじめ渡せば、1回の分割で同じ情報にアクセスできる**ので、木が浅くて済み、データ効率が上がります。

意味的にも自然です。「環境意識が高い人」も「補助金がある人」もそれぞれEV購入に前向きですが、**両方揃った人は単なる足し算以上に買う**——それが交互作用です。

**用語補足**：*交互作用（interaction）* = 2つの変数の効果が、それぞれ単独の効果の和では説明できない状態のこと。

In [ ]:
# Interaction признаки на основе EDA
X['Env_Concern_x_Subsidy'] = X['Environmental_Concern_Level'] * X['LE_Subsidy_Available']
test['Env_Concern_x_Subsidy'] = test['Environmental_Concern_Level'] * test['LE_Subsidy_Available']

X['Env_Concern_x_Income'] = X['Environmental_Concern_Level'] * X['Annual_Income_USD']
test['Env_Concern_x_Income'] = test['Environmental_Concern_Level'] * test['Annual_Income_USD']

X['Subsidy_x_Income'] = X['LE_Subsidy_Available'] * X['Annual_Income_USD']
test['Subsidy_x_Income'] = test['LE_Subsidy_Available'] * test['Annual_Income_USD']

X['Env_Concern_x_Commute'] = X['Environmental_Concern_Level'] * X['Daily_Commute_km']
test['Env_Concern_x_Commute'] = test['Environmental_Concern_Level'] * test['Daily_Commute_km']

X['Home_Charging_x_Subsidy'] = X['LE_Home_Charging_Possible'] * X['LE_Subsidy_Available']
test['Home_Charging_x_Subsidy'] = test['LE_Home_Charging_Possible'] * test['LE_Subsidy_Available']

X['Range_Anxiety_x_Subsidy'] = X['LE_Range_Anxiety_Level'] * X['LE_Subsidy_Available']
test['Range_Anxiety_x_Subsidy'] = test['LE_Range_Anxiety_Level'] * test['LE_Subsidy_Available']

X['Income_per_Concern'] = X['Annual_Income_USD'] / (X['Environmental_Concern_Level'] + 1)
test['Income_per_Concern'] = test['Annual_Income_USD'] / (test['Environmental_Concern_Level'] + 1)

X['Commute_per_Concern'] = X['Daily_Commute_km'] / (X['Environmental_Concern_Level'] + 1)
test['Commute_per_Concern'] = test['Daily_Commute_km'] / (test['Environmental_Concern_Level'] + 1)

print(f"✅ Добавлено interaction признаков")
print(f"X shape: {X.shape}")

## 🔢 5. DIGIT FEATURES — ловим артефакты синтетики

What are digit features?
We extract each digit of numerical features at positions -4 to 3: digit_k(x) = (x // 10^k) % 10. For 7 numeric columns this gives 56 lightweight int8 features.

Why they work:
Synthetic data often contains artifacts in digits (repeated decimals, rounding patterns). Trees struggle to capture these micro-patterns from raw values; explicit digits make them usable.

### セル解説：digit features（桁の抽出）

**何をしているか**：7つの数値列それぞれについて、`(x // 10**k) % 10` で **k = -4 〜 3 の各桁の数字**を取り出し、`int8` の新しい列にしています。7列 × 8桁 = **56個の新特徴量**。

`k` が負なら小数部の桁（`k=-1` で小数第1位）、正なら整数部の桁（`k=1` で十の位）です。

**なぜそうするのか**：直前の原著のmarkdownセルにある通り、**合成データには生成過程の指紋（artifact）が桁に残る**ことがあります。四捨五入のパターン、繰り返す小数、特定の桁だけ分布が偏る、など。

木モデルは「値が12345.67より大きいか」という**閾値による分割**しかできないので、「小数第2位が7かどうか」のような**桁レベルの周期的パターン**を捉えるには膨大な分割が必要で、事実上不可能です。桁を明示的に列として渡してあげれば、1回の分割で使えるようになります。

**効果は +0.00143 と、このnotebook最大の改善**でした。ただしこれは**合成データ特有の技**で、実務の生データではほぼ効きません（むしろノイズになります）。Playground Series を解くときの定石として覚えておく類のテクニックです。

**用語補足**：`//` は切り捨て除算、`%` は剰余。`int8` は -128〜127 の小さな整数型で、0〜9しか入らない桁データにはこれで十分。メモリを1/8に節約できます。

In [ ]:
# Числовые признаки для извлечения цифр
digit_num_cols = [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km',
    'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
]

# Извлекаем цифры в позициях от -4 до 3
for col in digit_num_cols:
    for k in range(-4, 4):
        new_col = f'{col}_digit{k}'
        X[new_col] = (X[col].fillna(0) // (10**k) % 10).astype('int8')
        test[new_col] = (test[col].fillna(0) // (10**k) % 10).astype('int8')

print(f"✅ Добавлено digit features: {len(digit_num_cols) * 8}")
print(f"X shape: {X.shape} | Test shape: {test.shape}")

## 📊 6. FREQUENCY ENCODING — частоты для ВСЕХ признаков

What:
Each value in every column is replaced by its normalized frequency across the combined train and test sets. Rare values get small frequencies, common values get large frequencies.

Why it works:
It gives XGBoost a different view of the data: how unusual or typical a value is. In synthetic datasets, generators often leave unusual distributions or rare combinations. Frequency features expose these patterns directly, without many sequential tree splits.

Effect:
OOF AUC improved from 0.94347 to 0.94466.

### セル解説：frequency encoding（頻度エンコーディング）

**何をしているか**：**すべての列**（digit featuresも含む）について、各値を「その値が train+test 全体の中で何割を占めるか」に置き換えた `_freq` 列を追加しています。

**なぜそうするのか**：モデルに**「この値は珍しいか、ありふれているか」という別の視点**を与えます。合成データでは、生成器が残す不自然な頻度分布（特定の値だけ異常に多い／少ない）が予測に効くことがあります。

木モデルがこれを自力で学ぶには「値がAなら…値がBなら…」と全ての値について分割を刻む必要があり非現実的ですが、頻度に変換すれば**1回の分割で「珍しい値かどうか」を判定できます**。

**効果は +0.00119** で、digit features に次ぐ改善でした。

**重要な注意点**：`pd.concat([X[column], test[column]])` と、**train と test を合わせて**頻度を計算しています。これは *transductive*（テストの入力分布を使う）なアプローチで、Kaggleでは一般的に認められていますが、**実務では使えません**（推論時に未来のデータ全体は手に入らないため）。またこれを CV の中でやると **fold間で情報が漏れる** ため、CV スコアが楽観的に出る危険があります。頻度は目的変数を使っていないので target encoding ほど危険ではありませんが、原理的にはリークの一種です。

**用語補足**：*target encoding* は「カテゴリを、そのカテゴリの目的変数の平均で置き換える」手法で、こちらは目的変数を使うため**必ずfold内で計算しないとリークします**。frequency encoding は目的変数を使わない、その安全な親戚です。

In [ ]:
# Считаем частоты по train+test для каждой колонки (включая digit и LE)
for column in X.columns:
    freq_map = pd.concat([X[column], test[column]], axis=0).value_counts(normalize=True).to_dict()
    X[f'{column}_freq'] = X[column].map(freq_map).astype('float32').values
    test[f'{column}_freq'] = test[column].map(freq_map).astype('float32').values

print(f"✅ Добавлено frequency features: {len([c for c in X.columns if c.endswith('_freq')])}")
print(f"X shape: {X.shape} | Test shape: {test.shape}")

## 7. 🧹 УДАЛЯЕМ OBJECT КОЛОНКИ

### セル解説：object型の列を削除

**何をしているか**：文字列型（`object`）のまま残っている列を落としています。

**なぜそうするのか**：XGBoost は数値しか受け付けません（`enable_categorical=True` を使わない限り）。前処理で残ってしまった文字列列があると学習時にエラーになるので、ここで確実に掃除しています。

`select_dtypes(include=['object'])` は「この型の列だけ選ぶ」というpandasの便利メソッドです。列名をハードコードするより、**型で選ぶ方が変更に強い**（新しい前処理を足しても壊れない）という利点があります。

In [ ]:
object_cols = X.select_dtypes(include=['object']).columns.tolist()
X = X.drop(columns=object_cols)
test = test.drop(columns=object_cols)

print(f"Удалено object: {len(object_cols)}")
print(f"X: {X.shape} | test: {test.shape}")

## 8. 🎯 ОТБОР ПРИЗНАКОВ — удаляем мусор


### セル解説：特徴量選択（定数列と完全相関列の除去）

**何をしているか**：2種類の「無駄な列」を落としています。

1. **定数列** — `nunique() == 1`、つまり全行が同じ値。分割に一切使えないので情報量ゼロ。
2. **完全相関列** — 相関係数の絶対値が **ちょうど 1.0** のペアの片方。片方が分かればもう片方も完全に決まるので、重複情報。

相関行列の**上三角だけ**（`np.triu(..., k=1)`）を見ているのがポイントで、これで (A,B) と (B,A) を二重に数えるのを防いでいます。

**なぜそうするのか**：結果は **OOF AUC 0.94488 → 0.94488（変化なし）、LB も変化なし、特徴量 154 → 76**。つまり**スコアは1ミリも変わらず、モデルが半分の大きさになった**ということです。

これは軽視されがちですが重要な成果です。学習が速くなれば実験を回せる回数が増え、モデルが単純になれば挙動が読みやすくなります。**「スコアが上がらないなら意味がない」ではなく、「同じスコアをより少ない材料で出せた」ことに価値がある**、という視点を持ってください。

最後の `del corr_matrix; gc.collect()` は、巨大な相関行列（154×154でも、列が増えれば二乗で膨らむ）をメモリから解放する定型句です。

In [ ]:
# Удаляем константные
const_cols = [c for c in X.columns if X[c].nunique() == 1]
X = X.drop(columns=const_cols)
test = test.drop(columns=const_cols)

# Удаляем коррелирующие (corr=1)
corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] == 1.0)]
X = X.drop(columns=to_drop)
test = test.drop(columns=to_drop)

print(f"✅ Удалено: {len(const_cols)} константных + {len(to_drop)} коррелирующих")
print(f"X: {X.shape} | test: {test.shape}")

del corr_matrix, upper_tri
gc.collect()

## 🚀 9. XGBOOST — BASELINE MODEL (10-fold, dynamic seeds)

🎯 Validation Strategy — 10-Fold CV with Dynamic Seeds

We use **10-fold stratified cross-validation** to get a more stable and honest estimate of model performance.  
On each fold, the random seed is **different** (`42 + fold`), which creates additional diversity across models and reduces the CV–LB gap.

| Setting | Value |
|---------|-------|
| Folds | 10 |
| Split type | StratifiedKFold |
| Shuffle | True |
| Base seed | 42 |
| Per-fold seed | 42 + fold (43, 44, ..., 52) |

This strategy is deliberately stronger than a single-seed 5-fold setup:  

- more training data per fold (90% train / 10% valid),  
- more robust OOF estimate,  
- lower variance across runs,  
- better generalization on the private leaderboard.  

We trust **OOF AUC** as the main optimization target — public LB is treated as a noisy sample.

### セル解説：XGBoost の学習（10-fold 層化交差検証）

**何をしているか**：目的変数を 0/1 に変換し、10分割の層化交差検証（StratifiedKFold）でXGBoostを学習します。各foldで、検証用のOOF予測とテスト予測（10分割の平均）を貯めていきます。

**パラメータの読み方**（初心者向け）：

| パラメータ | 値 | 意味 |
|---|---|---|
| `learning_rate` | 0.005 | 1本の木が結果を動かす幅。**小さいほど慎重に学び、精度は上がるが木が大量に必要** |
| `n_estimators` | 10000 | 木の最大本数。低い学習率とセットで使う |
| `early_stopping_rounds` | 500 | 検証AUCが500本連続で改善しなければ打ち切り。**「10000本」は上限であって、実際は自動で止まる** |
| `max_depth` | 7 | 木の深さ。深いほど複雑な交互作用を表現できるが過学習しやすい |
| `min_child_weight` | 10 | 葉に必要な最小サンプル重み。**大きいほど、たまたま数件だけ当てはまるノイズ的な分割を禁止できる** |
| `subsample` / `colsample_bytree` | 0.9 / 0.9 | 各木で使う行/列の割合。木ごとに違うデータを見せて多様性を作る |
| `reg_alpha` / `reg_lambda` | 0.071 / 2.0 | L1/L2正則化。L1は不要な特徴を0に押しやり、L2は葉の値が極端になるのを防ぐ |
| `max_bin` | 1024 | ヒストグラム分割の解像度。細かいほど良い分割点を見つけられるが遅い |

**なぜこの設計か**：**低い学習率 × 多い木の本数 × 強い正則化 × early stopping** は、GBDT（勾配ブースティング決定木）で精度を出すときの王道の組み合わせです。「ゆっくり、たくさん、慎重に、そして止め時は検証データに決めさせる」。early stopping があるので `n_estimators=10000` は危険ではありません。

**⚠️ ここで原著の記述とコードが食い違っています。** 上の説明セルは「fold ごとに `42 + fold` という**動的シード**を使う」と書いていますが、実際のコードは：

```python
current_seed = 42        # ← fold に依存していない
params['random_state'] = current_seed
```

**全fold同じシード42**です。コメントは「Динамический seed: 42 + fold」と書いてあるのに、実装がそうなっていません。おそらく実験の途中で戻したまま説明を更新し忘れたのでしょう。

原著自身が「固定シード vs 動的シードは差がなかった（±0.00001）」と実験結果を載せているので**結論は変わりません**が、これは重要な教訓です：**notebookの説明文と実装は一致しているとは限りません。数字を鵜呑みにする前に、必ずコードを読んでください。** `early_stopping_rounds` も、説明表では 700 なのに実装は 500 です。

**用語補足**：*OOF（Out-Of-Fold）予測* とは、「そのデータが検証側に回ったfoldのモデルによる予測」を全データ分集めたもの。自分自身を学習に使っていないので、**過学習していない公正な予測**としてスタッキングの入力に使えます。*層化（Stratified）* とは、各foldで正例と負例の比率を元データと同じに保つこと。

In [ ]:
import xgboost as xgb

# Конвертируем таргет в 0/1
y = y.map({'No': 0, 'Yes': 1})

# Параметры: старые + 10 фолдов + динамические сиды
params = {
    'objective': 'binary:logistic',       # Binary classification task
    'eval_metric': 'auc',                 # Evaluation metric: ROC AUC
    'tree_method': 'hist',                # Fast histogram-based tree builder
    'learning_rate': 0.005,               # Very low LR for deep, stable learning (like LGB 0.94612)
    'max_depth': 7,                       # Tree depth: deep enough for interactions, not too complex
    'min_child_weight': 10,               # Stronger regularization: prevents overfitting on noise
    'subsample': 0.9,                     # 90% of rows per tree: more data, less variance
    'colsample_bytree': 0.9,              # 90% of features per tree: more signal per split
    'device': 'cuda',                     # GPU acceleration: 3-5x faster training
    'reg_alpha': 0.071,                   # L1 regularization: feature selection pressure
    'reg_lambda': 2.0,                    # L2 regularization: prevents large leaf weights
    'max_bin': 1024,                      # More bins: better split resolution (like LGB)
    'n_estimators': 10000,                # Many trees: model has room to learn with low LR
    'early_stopping_rounds': 500,         # High patience: gives the model time to improve
    'nthread': -1,                        # Use all CPU threads
    'deterministic_histogram': True,      # Reproducible results across runs
}
# StratifiedKFold — 10 фолдов
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Массивы для сохранения
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))
feature_importance = np.zeros(X.shape[1])

log("Starting 10-fold CV...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Динамический seed: 42 + fold (43, 44, 45...)
    current_seed = 42
    params['random_state'] = current_seed
    params['seed'] = current_seed
    
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(test)[:, 1] / skf.n_splits
    
    feature_importance += model.feature_importances_ / skf.n_splits
    
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    h2(f"Fold {fold}/10")
    md_metric("Fold AUC", f"{fold_auc:.5f}")
    md_metric("Seed", str(current_seed))
    md_metric("Elapsed", f"{time.perf_counter() - T0:.1f}s")

# Overall OOF score
oof_auc = roc_auc_score(y, oof_preds)
h2("🏆 Final OOF Performance")
md_metric("OOF AUC", f"{oof_auc:.5f}")

# Feature importance
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

h2("📊 Top 15 Features")
pretty_df(importance_df.head(15))

## 💾 10. SAVE PREDICTIONS

### セル解説：OOF/テスト予測の保存

**何をしているか**：OOF予測とテスト予測を `.npy`（NumPyのバイナリ形式）で保存しています。

**なぜそうするのか**：**後でスタッキング（stacking）やブレンドに使うため**です。複数のモデルのOOF予測を並べて second-level モデルの入力にする、あるいは単純にランク平均で混ぜる——どちらにも、この2つの配列が必要です。

原著の「Next Steps」に「Level 2 Stacking」「Blending with LightGBM/CatBoost」とあるのは、まさにこれを使う計画です。

**保存が必須なのはOOFの方**です。テスト予測だけならモデルを再実行すれば作れますが、OOF予測を作るには10-fold全体を回し直す必要があり、しかも**同じfold分割**でなければ他モデルと混ぜられません。だから作った時点で保存します。

なお本日のS6E9で最高スコア（LB 0.94633）を出しているnotebookは、まさにこの「揃ったfold分割のOOFを共有する」ことだけを目的にしたものでした。`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` に全モデルを統一して、コミュニティ全体がスタッキングできるようにする——**分割の統一そのものが価値になる**という良い例です。

In [ ]:
np.save('oof_preds_base.npy', oof_preds)
np.save('test_preds_base.npy', test_preds)

h2("💾 Save Predictions")
info_card("✅ Predictions Saved", 
          f"OOF: {oof_preds.shape} | Test: {test_preds.shape} | OOF AUC: {oof_auc:.5f}", 
          style="vi")

## 📤 11. CREATE SUBMISSION

### セル解説：提出ファイルの作成

**何をしているか**：test.csv の `id` 列と、10-foldの平均テスト予測を並べて `submission.csv` を書き出します。

**なぜそうするのか**：`test_preds` は学習ループの中で `+= model.predict_proba(test)[:, 1] / skf.n_splits` と累積されていました。つまり**10個のモデルの予測の単純平均**です。

これは *bagging*（バギング）の効果を持ちます。各foldのモデルは訓練データの9割ずつ違うものを見ているので、それぞれ少しずつ違う誤りをします。平均するとその誤りが打ち消し合い、**単一モデルより安定した予測**になります。CV の副産物として無料でアンサンブルが手に入る、というのが k-fold の隠れた利点です。

AUCは順位しか見ないので、確率値を平均するか順位を平均するかで結果は変わり得ますが、同じアーキテクチャ・同じ特徴量のモデル同士なら確率スケールが揃っているので、単純平均で問題ありません。（**異なるモデル間**を混ぜるときは、スケールの違いを無視できるランク平均のほうが安全です。）

In [ ]:
submission = pd.DataFrame({
    'id': pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv')['id'],
    'Will_Buy_EV': test_preds
})

submission.to_csv('submission.csv', index=False)

h2("📤 Submission Created")
info_card("✅ Submission Saved", f"Shape: {submission.shape}", style="vi")
pretty_df(submission.head(10))

## 📋 12. FUTURE PLAN

## ✅ Completed
- Baseline XGBoost model (10-fold CV)
- Final AUC: 0.94488 (OOF) / 0.94460 (LB)
- Label Encoding for categorical features
- Digit Features (+0.00143)
- Frequency Encoding (+0.00119)
- Hyperparameter Upgrade (+0.00022)
- Feature Selection: 154 → 76 features

## 🔥 Next Steps
- Secret Recipe Features (buy_score, worry_score from Chris Deotte)
- Target Encoding with m-schedule (5, 15, 80)
- Pseudo Labeling
- Level 2 Stacking
- Blending with LightGBM/CatBoost